<a href="https://colab.research.google.com/github/despoileralert/DataScience_MachineLearning_Notebooks/blob/main/DT_Ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import warnings
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import (
    RFE,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    f_classif,
    mutual_info_classif,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, HalvingGridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Inital Setup

In [2]:
# drive mount
from google.colab import drive
drive.mount('/content/drive')

shared_folder_path = '/content/drive/MyDrive/time-series-human'

# Example: List the files inside the shared folder
import os
print(os.listdir(shared_folder_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['features.txt', 'activity_labels.txt', 'README.txt', 'sampler.py', 'features_info.txt', 'view.ipynb', 'RawData', 'Train', 'Model Training', 'Test', 'feature_extractor.py']


In [3]:
df = pd.read_csv('/content/drive/MyDrive/time-series-human/Train/X_train.txt', sep=r'\s+', header=None)

new_cols = []
with open('/content/drive/MyDrive/time-series-human/features.txt', 'r') as f:
    for line in f:
        new_cols.append(line.strip())
df.columns = new_cols
df

,tBodyAcc-Mean-1,tBodyAcc-Mean-2,tBodyAcc-Mean-3,tBodyAcc-STD-1,tBodyAcc-STD-2,tBodyAcc-STD-3,tBodyAcc-Mad-1,tBodyAcc-Mad-2,tBodyAcc-Mad-3,tBodyAcc-Max-1,...,fBodyGyroJerkMag-MeanFreq-1,fBodyGyroJerkMag-Skewness-1,fBodyGyroJerkMag-Kurtosis-1,tBodyAcc-AngleWRTGravity-1,tBodyAccJerk-AngleWRTGravity-1,tBodyGyro-AngleWRTGravity-1,tBodyGyroJerk-AngleWRTGravity-1,tXAxisAcc-AngleWRTGravity-1,tYAxisAcc-AngleWRTGravity-1,tZAxisAcc-AngleWRTGravity-1
0,0.043580,-0.005970,-0.035054,-0.995381,-0.988366,-0.937382,-0.995007,-0.988816,-0.953325,-0.794796,...,-0.012236,-0.314848,-0.713308,-0.112754,0.030400,-0.464761,-0.018446,-0.841559,0.179913,-0.051718
1,0.039480,-0.002131,-0.029067,-0.998348,-0.982945,-0.971273,-0.998702,-0.983315,-0.974000,-0.802537,...,0.202804,-0.603199,-0.860677,0.053477,-0.007435,-0.732626,0.703511,-0.845092,0.180261,-0.047436
2,0.039978,-0.005153,-0.022651,-0.995482,-0.977314,-0.984760,-0.996415,-0.975835,-0.985973,-0.798477,...,0.440079,-0.404427,-0.761847,-0.118559,0.177899,0.100699,0.808529,-0.849230,0.180610,-0.042271
3,0.039785,-0.011809,-0.028916,-0.996194,-0.988569,-0.993256,-0.996994,-0.988526,-0.993135,-0.798477,...,0.430891,-0.138373,-0.491604,-0.036788,-0.012892,0.640011,-0.485366,-0.848947,0.181907,-0.040826
4,0.038758,-0.002289,-0.023863,-0.998241,-0.986774,-0.993115,-0.998216,-0.986479,-0.993825,-0.801982,...,0.137735,-0.366214,-0.702490,0.123320,0.122542,0.693578,-0.615971,-0.848164,0.185124,-0.037080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7762,0.048048,-0.042445,-0.065884,-0.195448,-0.278326,-0.219954,-0.282233,-0.305861,-0.357803,0.267874,...,-0.008381,-0.596760,-0.879026,-0.190437,0.829718,0.206972,-0.425619,-0.792292,0.238580,0.056020
7763,0.037639,0.006430,-0.044345,-0.235372,-0.302680,-0.232843,-0.322483,-0.354464,-0.345592,0.181271,...,0.209452,-0.404418,-0.684496,0.064907,0.875679,-0.879033,0.400219,-0.772288,0.252653,0.056252
7764,0.037451,-0.002724,0.021009,-0.218281,-0.378082,-0.076950,-0.304446,-0.400661,-0.193071,0.113141,...,0.237003,0.000207,-0.317314,0.052806,-0.266724,0.864404,0.701169,-0.779566,0.249121,0.047071
7765,0.044011,-0.004536,-0.051242,-0.219202,-0.383350,-0.081035,-0.310419,-0.380233,-0.201007,0.166671,...,0.069366,0.037919,-0.356579,-0.101360,0.700740,0.936674,-0.589479,-0.785603,0.246409,0.031700


# Cross Validation Step

In [5]:
#Firstly we handle the train test split

base = "/content/drive/MyDrive/time-series-human"

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

trainingset = pd.read_csv(os.path.join(base, "Train", "features_train.csv")) #Train set
testingset = pd.read_csv(os.path.join(base, "Test", "features_test.csv"))   #Test set

X_train, X_test = trainingset.iloc[:, 3:], testingset.iloc[:, 3:]
y_train, y_test = (trainingset.iloc[:, 2]).to_numpy(), testingset.iloc[:, 2].to_numpy()
Xgroups, ygroups = (trainingset.iloc[:, 0]).to_numpy(), testingset.iloc[:, 0].to_numpy()

print(np.unique(Xgroups))
print(np.unique(ygroups))
print(type(X_train), type(y_train))

#Initialize CV method
sgkf = StratifiedGroupKFold(n_splits=5, shuffle = True, random_state = 42)
sgkf.get_n_splits()

groups = Xgroups
"""print("Xtrain is ", X_train)
print("y_train is ", y_train)"""
for i, (train_index, val_index) in enumerate(sgkf.split(X_train, y_train, Xgroups)):
    print(f"Fold {i}:")
    print(f"  Train: index length = { len(train_index)}")
    print(f"         group={np.unique(groups[train_index])}")
    print(f"  Validation:  index length = {len(val_index)}")
    print(f"         group={np.unique(groups[val_index])}")

[ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
[ 2  4  9 10 12 13 18 20 24]
<class 'pandas.core.frame.DataFrame'> <class 'numpy.ndarray'>
Fold 0:
  Train: index length = 6239
         group=[ 1  3  5  6  8 11 14 15 16 17 21 22 23 26 27 29 30]
  Validation:  index length = 1528
         group=[ 7 19 25 28]
Fold 1:
  Train: index length = 6184
         group=[ 1  3  5  6  7  8 11 14 15 19 21 22 23 25 26 28 29]
  Validation:  index length = 1583
         group=[16 17 27 30]
Fold 2:
  Train: index length = 6024
         group=[ 3  5  6  7 14 16 17 19 21 22 25 26 27 28 29 30]
  Validation:  index length = 1743
         group=[ 1  8 11 15 23]
Fold 3:
  Train: index length = 6363
         group=[ 1  5  6  7  8 11 15 16 17 19 21 23 25 26 27 28 30]
  Validation:  index length = 1404
         group=[ 3 14 22 29]
Fold 4:
  Train: index length = 6258
         group=[ 1  3  7  8 11 14 15 16 17 19 22 23 25 27 28 29 30]
  Validation:  index length = 1509
         group=[ 5  6 21 26]

In [8]:
# We measure the performance of base DT and Random Foreset Models.

X_orig = X_train.loc[:, ~X_train.columns.str.startswith("wavelet")]

models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            n_jobs=-1, random_state=42),
}

featuresets = {"X_orig": X_orig, "X_train": X_train}
results = []

# The Matthews correlation coefficient is used in machine learning as a measure of the quality of binary and multiclass classifications.
# It takes into account true and false positives and negatives and is generally regarded as a balanced measure which can be used even if the classes are of very different sizes.
# The MCC is in essence a correlation coefficient value between -1 and +1.
# A coefficient of +1 represents a perfect prediction, 0 an average random prediction and -1 an inverse prediction.
# The statistic is also known as the phi coefficient. [source: Wikipedia]

scoringmetrics = ["balanced_accuracy", "f1_macro", "matthews_corrcoef"]
#Train on original features + extended features
for feature in featuresets:
  for estname, estimator in models.items():
    s = cross_validate(estimator, featuresets[feature], y_train,
                      groups = Xgroups,
                      scoring = scoringmetrics,
                      cv = sgkf,
                      verbose = 5)
    entry = {"featureset": feature, "model": estname}
    for i in s:
      entry[i] = np.mean(s[i])
    entry["model"] = estname
    results.append(entry)
  print(results)

finaltable = pd.DataFrame(results).round(3)
finaltable

[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.682) matthews_corrcoef: (test=0.769) total time=   9.7s
[CV] END  balanced_accuracy: (test=0.746) f1_macro: (test=0.732) matthews_corrcoef: (test=0.805) total time=   6.8s
[CV] END  balanced_accuracy: (test=0.744) f1_macro: (test=0.741) matthews_corrcoef: (test=0.853) total time=   9.1s
[CV] END  balanced_accuracy: (test=0.648) f1_macro: (test=0.652) matthews_corrcoef: (test=0.784) total time=   9.0s
[CV] END  balanced_accuracy: (test=0.620) f1_macro: (test=0.604) matthews_corrcoef: (test=0.793) total time=   9.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   43.6s finished


[CV] END  balanced_accuracy: (test=0.764) f1_macro: (test=0.743) matthews_corrcoef: (test=0.853) total time=  52.3s
[CV] END  balanced_accuracy: (test=0.827) f1_macro: (test=0.833) matthews_corrcoef: (test=0.909) total time=  53.9s
[CV] END  balanced_accuracy: (test=0.822) f1_macro: (test=0.830) matthews_corrcoef: (test=0.950) total time=  51.8s
[CV] END  balanced_accuracy: (test=0.766) f1_macro: (test=0.763) matthews_corrcoef: (test=0.892) total time=  54.1s
[CV] END  balanced_accuracy: (test=0.745) f1_macro: (test=0.762) matthews_corrcoef: (test=0.848) total time=  56.4s
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(8.707699537277222), 'score_time': np.float64(0.014462852478027343), 'test_balanced_accuracy': np.float64(0.6891547262951427), 'test_f1_macro': np.float64(0.6822009341250407), 'test_matthews_corrcoef': np.float64(0.8006883543170892)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(53.552759885787964), 'score_time': np

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  4.5min finished


[CV] END  balanced_accuracy: (test=0.747) f1_macro: (test=0.723) matthews_corrcoef: (test=0.790) total time=  19.7s
[CV] END  balanced_accuracy: (test=0.760) f1_macro: (test=0.747) matthews_corrcoef: (test=0.812) total time=  15.6s
[CV] END  balanced_accuracy: (test=0.820) f1_macro: (test=0.821) matthews_corrcoef: (test=0.887) total time=  17.9s
[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.686) matthews_corrcoef: (test=0.756) total time=  22.5s
[CV] END  balanced_accuracy: (test=0.754) f1_macro: (test=0.739) matthews_corrcoef: (test=0.798) total time=  19.1s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  1.6min finished


[CV] END  balanced_accuracy: (test=0.811) f1_macro: (test=0.803) matthews_corrcoef: (test=0.855) total time= 1.3min
[CV] END  balanced_accuracy: (test=0.836) f1_macro: (test=0.847) matthews_corrcoef: (test=0.911) total time= 1.2min
[CV] END  balanced_accuracy: (test=0.865) f1_macro: (test=0.865) matthews_corrcoef: (test=0.958) total time= 1.2min
[CV] END  balanced_accuracy: (test=0.817) f1_macro: (test=0.814) matthews_corrcoef: (test=0.894) total time= 1.3min
[CV] END  balanced_accuracy: (test=0.838) f1_macro: (test=0.820) matthews_corrcoef: (test=0.849) total time= 1.3min
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(8.707699537277222), 'score_time': np.float64(0.014462852478027343), 'test_balanced_accuracy': np.float64(0.6891547262951427), 'test_f1_macro': np.float64(0.6822009341250407), 'test_matthews_corrcoef': np.float64(0.8006883543170892)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(53.552759885787964), 'score_time': np

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  6.3min finished


,featureset,model,fit_time,score_time,test_balanced_accuracy,test_f1_macro,test_matthews_corrcoef
0,X_orig,Decision tree,8.708,0.014,0.689,0.682,0.801
1,X_orig,Random forest,53.553,0.138,0.785,0.786,0.890
2,X_train,Decision tree,18.947,0.019,0.754,0.743,0.809
3,X_train,Random forest,75.842,0.144,0.833,0.830,0.894


# Correlation Filter + Dimensionality Reduction + Model Training

In [9]:
!pip install feature_engine

Various transform steps to be used in the pipeline later on

In [10]:
from sklearn.feature_selection import SelectFromModel
from feature_engine.selection import DropCorrelatedFeatures, SmartCorrelatedSelection
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

corrFilter = SmartCorrelatedSelection()
selector = SelectFromModel(estimator=ExtraTreesClassifier(random_state=42))

#Other DimReduction transformers to compare against
KPCAtransform = KernelPCA(random_state=42, kernel='rbf')
FAtransform = FactorAnalysis(random_state=42)
ICAtransform = FastICA(random_state=42)
LDAtransform = LinearDiscriminantAnalysis()
SVDtransform = TruncatedSVD(random_state=42)
GRPtransform = GaussianRandomProjection(random_state=42)

In [14]:
n_components_options = [50, 100, 200]

corr_params = {
    "corr__threshold": [0.9, 0.95],
    "corr__method": ["pearson", "spearman"],
    "corr__selection_method": ["variance"],
}

rf_params = {
    "classify__n_estimators": [200, 400],
    "classify__max_features": ["sqrt", "log2"],
    "classify__min_samples_leaf": [3, 5],
}

transform_param_grid = [
    {
        "reduce_dim": [KPCAtransform],
        "reduce_dim__n_components": n_components_options,
        "reduce_dim__kernel": ["rbf"],
    },
    {
        "reduce_dim": [FAtransform, GRPtransform],
        "reduce_dim__n_components": n_components_options,
    },
    {
        "reduce_dim": [ICAtransform],
        "reduce_dim__n_components": n_components_options,
        "reduce_dim__whiten": ['unit-variance'],
        "reduce_dim__fun": ['logcosh', 'exp', 'cube'],
        "reduce_dim__whiten_solver": ['svd'],
    },
    {
        "reduce_dim": [LDAtransform],
        "reduce_dim__n_components": [11],
        "reduce_dim__solver": ["eigen"],
        "reduce_dim__shrinkage": ["auto"]
    },
    {
        "reduce_dim": [SVDtransform],
        "reduce_dim__n_components": n_components_options,
    },
    {
        "reduce_dim": [selector],
        "reduce_dim__threshold": [-np.inf],                  # with -inf, max_features = exact count kept
        "reduce_dim__max_features": [100, 200, 300, 400],
    }
]
reducer_labels = ["KPCA", "FA", "Model Selection", "ICA", "LDA", "SVD", "GRP"]

fullparamgrid = [{**corr_params, **g, **rf_params} for g in transform_param_grid]

RFpipe = Pipeline(
    [
        ("corr", corrFilter),
        ("reduce_dim", "passthrough"),
        ("classify", RandomForestClassifier(random_state = 42)),
    ],
    memory = "pipeline_cache"
)


print(fullparamgrid)
RFmodel = GridSearchCV(RFpipe, param_grid=fullparamgrid, cv=sgkf, scoring="f1_macro", n_jobs=1, refit=True, verbose=10)
RFmodel.fit(X_train, y_train, groups = Xgroups)

[{'corr__threshold': [0.9, 0.95], 'corr__method': ['pearson', 'spearman'], 'corr__selection_method': ['variance'], 'reduce_dim': [KernelPCA(kernel='rbf', random_state=42)], 'reduce_dim__n_components': [50, 100, 200], 'reduce_dim__kernel': ['rbf'], 'classify__n_estimators': [200, 400], 'classify__max_features': ['sqrt', 'log2'], 'classify__min_samples_leaf': [3, 5]}, {'corr__threshold': [0.9, 0.95], 'corr__method': ['pearson', 'spearman'], 'corr__selection_method': ['variance'], 'reduce_dim': [FactorAnalysis(random_state=42), GaussianRandomProjection(random_state=42)], 'reduce_dim__n_components': [50, 100, 200], 'classify__n_estimators': [200, 400], 'classify__max_features': ['sqrt', 'log2'], 'classify__min_samples_leaf': [3, 5]}, {'corr__threshold': [0.9, 0.95], 'corr__method': ['pearson', 'spearman'], 'corr__selection_method': ['variance'], 'reduce_dim': [FastICA(random_state=42)], 'reduce_dim__n_components': [50, 100, 200], 'reduce_dim__whiten': ['unit-variance'], 'reduce_dim__fun': 

KeyboardInterrupt: 